In [1]:
from collections import defaultdict

import torch

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import DataCollatorForSeq2Seq

# 1. Model Loading

In [2]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

# model = T5ForConditionalGeneration.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
# )

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to("cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
print("Model loaded on:", device)

Model loaded on: cuda


In [4]:
model.eval()

prompt = "What can you do"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Make a sand castle


In [ ]:
sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
inputs = tokenizer(sample_input, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

No


# 2. Model Inspection

In [ ]:
print(model)

In [ ]:
encoder_block_0 = model.encoder.block[0]
print(encoder_block_0)

In [ ]:
decoder_block_0 = model.decoder.block[0]
print(decoder_block_0)

In [ ]:
lm_head = model.lm_head
print(lm_head)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

# 3. Full Training

In [ ]:
param_stats = defaultdict(int)

for name, param in model.named_parameters():
    param_stats[name.split('.')[0]] += param.numel()

for k, v in param_stats.items():
    print(f"{k}: {v:,}")

In [ ]:
for name, param in model.named_parameters():
    if "DenseReluDense" in name:
        print("FFN:", name, param.numel())
    elif "SelfAttention" in name or "EncDecAttention" in name:
        print("ATTN:", name, param.numel())

In [ ]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = False


def enable_ffn_training(model):
    """
    Enable training only for FFN (DenseReluDense) layers
    in both encoder and decoder.
    """
    for name, module in model.named_modules():
        if module.__class__.__name__ == "T5DenseGatedActDense":
            for p in module.parameters():
                p.requires_grad = True

def print_trainable_params(model):
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name)

def count_parameters(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    return total, trainable


In [ ]:
freeze_all_params(model)
enable_ffn_training(model)
sum(p.requires_grad for p in model.parameters())

In [ ]:
print_trainable_params(model)

In [ ]:
total_params, trainable_params = count_parameters(model)

percent = 100 * trainable_params / total_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {percent:.2f}%")

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

In [ ]:
sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
inputs = tokenizer(sample_input, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Demo training

In [ ]:
import torch

from torch.utils.data import Dataset
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from torch.optim import AdamW

In [ ]:
data = [
    {
        "instruction": "Question: Does aspirin reduce fever? Answer yes or no.",
        "output": "Yes, aspirin can reduce fever."
    },
    {
        "instruction": "Question: Is vitamin C a cure for cancer? Answer yes or no.",
        "output": "No, vitamin C is not a cure for cancer."
    }
]

In [ ]:
class SimpleT5Dataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        enc = self.tokenizer(
            item["instruction"],
            truncation=True,
            padding=False
        )

        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                item["output"],
                truncation=True,
                padding=False
            )["input_ids"]

        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in labels
        ]

        return {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "labels": torch.tensor(labels)
        }


In [ ]:
dataset = SimpleT5Dataset(data, tokenizer)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100
)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator
)


In [ ]:
model = model.float().to("cuda")

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to("cuda") for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()
        print(f"Step {step} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} average loss: {avg_loss:.4f}")


# 4. LoRA Training

In [ ]:
import math

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Apply LoRA to FFNs

In [ ]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = False

def apply_lora_to_ffn(model, r=8, alpha=1.0):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)

def merge_lora_to_linear(model):
    for name, module in model.named_modules():
        # Only target LoRALinear
        if isinstance(module, LoRALinear):
            # Create a new nn.Linear with the same shape
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            # Copy base weight + LoRA contribution
            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            # Replace LoRALinear in the parent module
            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.A = nn.Linear(in_dim, r, bias=False)
        self.B = nn.Linear(r, out_dim, bias=False)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

In [ ]:
rank = 16
alpha = 32

freeze_all_params(model)
apply_lora_to_ffn(model, r=rank, alpha=alpha)

for name, param in model.named_parameters():
    if "A.weight" in name or "B.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters: {total:,}")
print(f"Trainable parameters (LoRA): {trainable:,}")
print(f"Frozen parameters: {frozen:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Total parameters: 250,821,888
Trainable parameters (LoRA): 3,244,032
Frozen parameters: 247,577,856
Trainable %: 1.2934%


## Dummy Training

In [ ]:
class DummyDataset(Dataset):
    def __init__(self, tokenizer):
        self.data = [
            {"input": "Question: Is the sky blue? Instruction: Answer in one sentence.",
             "output": "Yes, the sky is blue."},
            {"input": "Question: Do cats bark? Instruction: Answer in one sentence.",
             "output": "No, cats do not bark."},
            {"input": "Question: Is water wet? Instruction: Answer in one sentence.",
             "output": "Yes, water is wet."}
        ]
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        input_enc = self.tokenizer(example["input"], return_tensors="pt", truncation=True, padding=False)
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(example["output"], return_tensors="pt", truncation=True, padding=False)["input_ids"]
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_enc["input_ids"].squeeze(0),
            "attention_mask": input_enc["attention_mask"].squeeze(0),
            "labels": labels.squeeze(0)
        }

train_dataset = DummyDataset(tokenizer)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
model.to("cuda")
model.train()

num_epochs = 3

for epoch in range(num_epochs):
    running_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        input_ids = batch["input_ids"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        labels = batch["labels"].to("cuda")

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        print(f"Step {step} | Loss: {loss.item():.4f}", end="\r")

    avg_loss = running_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} average loss: {avg_loss:.4f}")


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Step 3 | Loss: 0.8254
Epoch 1 average loss: 1.1088
Step 3 | Loss: 0.9392
Epoch 2 average loss: 0.8231
Step 3 | Loss: 0.9052
Epoch 3 average loss: 0.5629


## Inference after training

In [ ]:
merge_lora_to_linear(model) # Always merge before inference

model.to("cuda")

model.eval()

sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
inputs = tokenizer(sample_input, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Aspirin is a drug that is used to treat heart attack.


In [ ]:
sample_input = "Question: Is water wet?? Instruction: Answer in one sentence."
inputs = tokenizer(sample_input, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Yes, water is wet.


# Training using PubMedQA DS

In [ ]:
# !pip install datasets

In [5]:
import math

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

## Preprocess dataset

In [6]:
dataset = load_dataset("pubmed_qa", "pqa_labeled")

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

In [13]:
dataset.column_names

{'train': ['pubid', 'question', 'context', 'long_answer', 'final_decision']}

In [14]:
dataset["train"][0]["final_decision"]

'yes'

In [15]:
print(set(dataset["train"]["final_decision"]))
from collections import Counter

Counter(dataset["train"]["final_decision"])


{'maybe', 'no', 'yes'}

In [ ]:
def preprocess_pubmedqa(example):
    context = " ".join(example["context"]["contexts"])

    input_text = (
        f"Question: {example['question']} "
        f"Context: {context} "
        f"Instruction: First answer yes, no, or maybe. "
        f"Then justify your answer briefly."
    )

    target_text = (
        f"Answer: {example['final_decision']}. "
        f"Explanation: {example['long_answer']}"
    )

    return {
        "input": input_text,
        "output": target_text
    }


In [ ]:
processed_ds = dataset.map(
    preprocess_pubmedqa,
    remove_columns=dataset["train"].column_names
)

In [ ]:
processed_ds["train"][0]

{'input': 'Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death? Context: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and 

In [ ]:
max_input_length = 512   # truncate long questions + context
max_target_length = 128  # truncate explanation if too long

def tokenize_for_t5(example):
    # Encode inputs
    input_enc = tokenizer(
        example["input"],
        truncation=True,
        padding="max_length",
        max_length=max_input_length,
    )

    # Encode targets (labels)
    target_enc = tokenizer(
        example["output"],
        truncation=True,
        padding="max_length",
        max_length=max_target_length
    )

    labels = target_enc["input_ids"]

    # Important: mask padding tokens
    labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]

    return {
        "input_ids": input_enc["input_ids"],
        "attention_mask": input_enc["attention_mask"],
        "labels": labels
    }

# Apply to dataset
tokenized_dataset = processed_ds.map(
    tokenize_for_t5,
    remove_columns=processed_ds["train"].column_names
)

In [ ]:
batch_size = 2  # small for testing, increase if GPU allows

def collate_fn(batch):
    input_ids = torch.tensor([item["input_ids"] for item in batch], dtype=torch.long)
    attention_mask = torch.tensor([item["attention_mask"] for item in batch], dtype=torch.long)
    labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

## Process model for training

In [ ]:
def freeze_all_params(model):
    for p in model.parameters():
        p.requires_grad = False

def apply_lora_to_ffn(model, r=8, alpha=1.0):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            continue  # Only wrap higher-level FFN
        if module.__class__.__name__ == "T5DenseGatedActDense":
            module.wi_0 = LoRALinear(module.wi_0, r=r, alpha=alpha)
            module.wi_1 = LoRALinear(module.wi_1, r=r, alpha=alpha)
            module.wo = LoRALinear(module.wo, r=r, alpha=alpha)

def merge_lora_to_linear(model):
    for name, module in model.named_modules():
        # Only target LoRALinear
        if isinstance(module, LoRALinear):
            # Create a new nn.Linear with the same shape
            new_linear = nn.Linear(
                module.base.in_features,
                module.base.out_features,
                bias=(module.base.bias is not None)
            ).to(module.base.weight.device)

            # Copy base weight + LoRA contribution
            new_linear.weight.data = module.base.weight.data + (module.B.weight @ module.A.weight) * module.scaling

            if module.base.bias is not None:
                new_linear.bias.data = module.base.bias.data

            # Replace LoRALinear in the parent module
            parent = module._modules
            for key, child in parent.items():
                if child is module:
                    parent[key] = new_linear
                    break


class LoRALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=1.0):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False  # Freeze base

        in_dim = base_linear.in_features
        out_dim = base_linear.out_features

        self.A = nn.Linear(in_dim, r, bias=False)
        self.B = nn.Linear(r, out_dim, bias=False)
        self.scaling = alpha / r

        # Initialize LoRA
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        return self.base(x) + self.scaling * self.B(self.A(x))

    # Expose weight/bias to satisfy HF generate()
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

In [ ]:
rank = 128
alpha = 256

# Freeze all parameters
freeze_all_params(model)

# Apply LoRA to FFN layers
apply_lora_to_ffn(model, r=rank, alpha=alpha)

# Make only LoRA parameters trainable
for name, param in model.named_parameters():
    if "A.weight" in name or "B.weight" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"Total parameters: {total:,}")
print(f"Trainable parameters (LoRA): {trainable:,}")
print(f"Frozen parameters: {frozen:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Total parameters: 250,821,888
Trainable parameters (LoRA): 3,244,032
Frozen parameters: 247,577,856
Trainable %: 1.2934%


In [ ]:
# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

from tqdm import tqdm

num_epochs = 3
model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    # Use tqdm for step progress
    loop = tqdm(enumerate(train_loader, 1), total=len(train_loader), desc=f"Epoch {epoch+1}")

    for step, batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # Update tqdm postfix with current loss
        loop.set_postfix(loss=loss.item())



    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")


Epoch 1: 100%|██████████| 500/500 [02:23<00:00,  3.49it/s, loss=2.36]


Epoch 1 average loss: 2.0855


Epoch 2: 100%|██████████| 500/500 [02:19<00:00,  3.58it/s, loss=2.28]


Epoch 2 average loss: 1.9952


Epoch 3: 100%|██████████| 500/500 [02:19<00:00,  3.59it/s, loss=1.32]

Epoch 3 average loss: 1.8749


In [ ]:
merge_lora_to_linear(model)
model.eval()

sample_input = "Question: Does aspirin reduce heart attack risk? Instruction: Answer in 2 sentences."
inputs = tokenizer(sample_input, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Answer: yes. Explanation: Aspirin reduces the risk of heart attack by reducing the amount of blood in the blood vessels.
